# Einlesen von LTOPs .mes-File und das gleichzeitige Umwandeln in ein JAG3d-Format

In [7]:
import os

## 1.0 Manuelles Einlesen und Exportieren

### 1.10 Einlesen des mes-File

In [8]:
# Datei einlesen
filename = "test_data/Umwandlung_ltop-jag3d/SATZ-1009.mes"  # Pfad anpassen

try:
    with open(filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()
except UnicodeDecodeError:
    with open(filename, 'r', encoding='latin-1') as f:
        lines = f.readlines()

print(lines)

['$$ME Satzmessung und Antennenmessung Station 1009 // Stand 08.09.2025\n', '** ----------------------------------------------------------------------------------------\n', '** ----------------------------------------------------------------------------------------\n', '** ----------------------------------------------------------------------------------------\n', '** Satzmessung (Hz-Winkel/V-Winkel/Ds) // Station 1009 -- Nr 1\n', '** ----------------------------------------------------------------------------------------\n', 'ST1009                                        1.5614\n', '** Satz 1\n', 'RI1011                       0.00000\n', 'RI1010                      44.24706\n', 'RI1014                       3.26626\n', 'RI8001                      42.57027\n', 'ZD1011                     114.31454                1.7844\n', 'ZD1010                     111.93651                1.8350\n', 'ZD1014                     116.86865                1.5772\n', 'DS1014                     121.934

### 1.20. Filterung der unnötigen Zeile

In [9]:
# Kommentarzeilen entfernen (beginnen mit **)
filtered_lines = []
for line in lines:
    line = line.strip()
    if line and not line.startswith('**') and not line.startswith('$$'):
        filtered_lines.append(line)

print(filtered_lines)

['ST1009                                        1.5614', 'RI1011                       0.00000', 'RI1010                      44.24706', 'RI1014                       3.26626', 'RI8001                      42.57027', 'ZD1011                     114.31454                1.7844', 'ZD1010                     111.93651                1.8350', 'ZD1014                     116.86865                1.5772', 'DS1014                     121.93427                1.5772', 'RI1011                       0.00000', 'RI1010                      44.24739', 'RI1014                       3.26517', 'RI8001                      42.56811', 'ZD1011                     114.31436                1.7844', 'ZD1010                     111.93731                1.8350', 'ZD1014                     116.86876                1.5772', 'DS1014                     121.93422                1.5772', 'RI1011                       0.00000', 'RI1010                      44.24701', 'RI1014                       3.26585', 'RI8001

### 1.30. Erstellen des Dict pro Station

In [10]:
# In Stationsblöcke aufteilen
stations = []
current_station = None

for line in filtered_lines:
    if line.startswith('ST'):
        # Neue Station gefunden - vorherige Station speichern
        if current_station is not None:
            stations.append(current_station)
        
        # Station parsen: ST<Punktname>    <Instrumentenhöhe>
        parts = line.split()
        station_name = parts[0][2:]  # ST entfernen
        instrument_height = float(parts[1]) if len(parts) > 1 else None
        
        current_station = {
            'station_name': station_name,
            'instrument_height': instrument_height,
            'measurements': []
        }
        
    elif current_station is not None and line:
        # Messung zur aktuellen Station hinzufügen
        current_station['measurements'].append(line)

# Letzte Station hinzufügen
if current_station is not None:
    stations.append(current_station)


print(stations)

[{'station_name': '1009', 'instrument_height': 1.5614, 'measurements': ['RI1011                       0.00000', 'RI1010                      44.24706', 'RI1014                       3.26626', 'RI8001                      42.57027', 'ZD1011                     114.31454                1.7844', 'ZD1010                     111.93651                1.8350', 'ZD1014                     116.86865                1.5772', 'DS1014                     121.93427                1.5772', 'RI1011                       0.00000', 'RI1010                      44.24739', 'RI1014                       3.26517', 'RI8001                      42.56811', 'ZD1011                     114.31436                1.7844', 'ZD1010                     111.93731                1.8350', 'ZD1014                     116.86876                1.5772', 'DS1014                     121.93422                1.5772', 'RI1011                       0.00000', 'RI1010                      44.24701', 'RI1014                       3.

### 1.40. Parsen der Messzile und Sortieren

In [11]:
# Vollständig separierte Gruppierung: RI getrennt, DS und ZD jeweils als eigene Gruppen
station_measurement_groups = {}

for station_index, station in enumerate(stations):
    station_name = station['station_name']
    station_height = station['instrument_height']
    
    for measurement_line in station['measurements']:
        # Messzeile parsen
        measurement_type = measurement_line[:2]
        target_point = measurement_line[2:12].strip()
        measurement_value = float(measurement_line[24:36].strip())
        mf = measurement_line[36:42].strip()
        instrument_height_pt = measurement_line[46:52].strip()
        target_height_pt = measurement_line[52:58].strip()

        # Falls keine explizite Instrumentenhöhe gefunden und DS/ZD, Station-Höhe verwenden
        if len(instrument_height_pt) == 0 and station_height is not None and measurement_type not in ["RI"]:
            instrument_height_pt = station_height

        ## mF Umwandeln von mm in m
        if measurement_type in ["DS"] and len(mf) > 0:
            mf = float(mf) / 1_000

        ## mF Umwandeln von cc in gon
        if measurement_type in ["ZD", "RI"] and len(mf) > 0:
            mf = float(mf) / 10_000

        # NEUE FILTERUNG: Eliminierte Messungen überspringen
        if isinstance(mf, (int, float)) and (mf == 9.999 or mf == 0.9999):
            continue  # Diese Messung überspringen - wird nicht zu measurement_groups hinzugefügt

        # Bestimme den Gruppierungsschlüssel basierend auf Messtyp
        if measurement_type == "RI":
            # RI: Getrennt nach Aufstellung (Index)
            group_key = f"{station_name}_{station_index}"
        elif measurement_type == "DS":
            # DS: Eine eigene Gruppe für alle DS-Messungen
            group_key = f"DS_{station_name}"
        elif measurement_type == "ZD":
            # ZD: Eine eigene Gruppe für alle ZD-Messungen  
            group_key = f"ZD_{station_name}"
        else:
            # Fallback für andere Messtypen
            group_key = f"{measurement_type}_{station_name}"
        
        # Gruppe initialisieren falls nötig
        if group_key not in station_measurement_groups:
            station_measurement_groups[group_key] = {}
        
        # Messtyp in der Gruppe initialisieren falls nötig
        if measurement_type not in station_measurement_groups[group_key]:
            station_measurement_groups[group_key][measurement_type] = []
        
        measurement_data = {
            'standpunkt': station_name,           # Standpunkt
            'anschlusspunkt': target_point,       # Anschlusspunkt
            'instrumentenhoehe': instrument_height_pt,  # Instrumentenhöhe
            'zielhoehe': target_height_pt,          # Zielhöhe
            'messwert': measurement_value,       # Messwert
            'standardabweichung': mf, # Standardabweichung
        }
        
        station_measurement_groups[group_key][measurement_type].append(measurement_data)

print(station_measurement_groups)


{'1009_0': {'RI': [{'standpunkt': '1009', 'anschlusspunkt': '1011', 'instrumentenhoehe': '', 'zielhoehe': '', 'messwert': 0.0, 'standardabweichung': ''}, {'standpunkt': '1009', 'anschlusspunkt': '1010', 'instrumentenhoehe': '', 'zielhoehe': '', 'messwert': 44.24706, 'standardabweichung': ''}, {'standpunkt': '1009', 'anschlusspunkt': '1014', 'instrumentenhoehe': '', 'zielhoehe': '', 'messwert': 3.26626, 'standardabweichung': ''}, {'standpunkt': '1009', 'anschlusspunkt': '8001', 'instrumentenhoehe': '', 'zielhoehe': '', 'messwert': 42.57027, 'standardabweichung': ''}, {'standpunkt': '1009', 'anschlusspunkt': '1011', 'instrumentenhoehe': '', 'zielhoehe': '', 'messwert': 0.0, 'standardabweichung': ''}, {'standpunkt': '1009', 'anschlusspunkt': '1010', 'instrumentenhoehe': '', 'zielhoehe': '', 'messwert': 44.24739, 'standardabweichung': ''}, {'standpunkt': '1009', 'anschlusspunkt': '1014', 'instrumentenhoehe': '', 'zielhoehe': '', 'messwert': 3.26517, 'standardabweichung': ''}, {'standpunkt'

### 1.50. Export des Messformat in ein JAG3D-Format

In [12]:
import os

# Verzeichnis erstellen falls nicht vorhanden
output_directory = "test_data/Umwandlung_ltop-jag3d/jag3d_export"
os.makedirs(output_directory, exist_ok=True)

exported_files = []

for group_key, measurement_types in station_measurement_groups.items():
    for measurement_type, measurements in measurement_types.items():
        if not measurements:
            continue
        
        # Korrigierte Dateiname-Erstellung - keine Dopplung bei DS/ZD
        if 'DS_' in group_key or 'ZD_' in group_key:
            filename = f"{group_key}.txt"  # Nur group_key verwenden
        else:
            filename = f"{measurement_type}_{group_key}.txt"  # Bei RI weiterhin mit Messtyp
        
        filepath = os.path.join(output_directory, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            # Header schreiben
            f.write(f"# JAG3D Import File - {measurement_type} Messungen\n")
            f.write(f"# Gruppe: {group_key}\n")
            f.write(f"# Format: <start point> <end point> <instrument height> <reflector height> <observation value> <uncertainty>\n")
            f.write(f"# Generiert aus LTOP mes-Datei\n")
            f.write("\n")
            
            for measurement in measurements:
                # Werte extrahieren - keine Defaults setzen
                start_point = measurement['standpunkt']
                end_point = measurement['anschlusspunkt'] 
                
                # Instrumentenhöhe: Nur verwenden wenn vorhanden und nicht leer
                instrument_height = measurement['instrumentenhoehe']
                if instrument_height == '' or instrument_height is None:
                    instrument_height = ''
                else:
                    instrument_height = str(instrument_height)
                
                # Zielhöhe: Nur verwenden wenn vorhanden und nicht leer
                reflector_height = measurement['zielhoehe']
                if reflector_height == '' or reflector_height is None:
                    reflector_height = ''
                else:
                    reflector_height = str(reflector_height)
                
                # Messwert
                observation_value = str(measurement['messwert'])
                
                # Unsicherheit: Nur verwenden wenn vorhanden und nicht leer
                uncertainty = measurement['standardabweichung']
                if uncertainty == '' or uncertainty is None:
                    uncertainty = ''
                else:
                    uncertainty = str(uncertainty)
                
                # Zeile zusammenstellen (Tabulator-getrennt, leere Felder bleiben leer)
                line = start_point + '\t' + end_point + '\t' + instrument_height + '\t' + reflector_height + '\t' + observation_value + '\t' + uncertainty + '\n'
                f.write(line)
        
        exported_files.append(filepath)
        print(f"✓ Exportiert: {filename} ({len(measurements)} Messungen)")

print(f"\n✓ Export abgeschlossen!")
print(f"✓ {len(exported_files)} Dateien erstellt")
print(f"✓ Verzeichnis: {output_directory}")


✓ Exportiert: RI_1009_0.txt (12 Messungen)
✓ Exportiert: ZD_1009.txt (24 Messungen)
✓ Exportiert: DS_1009.txt (15 Messungen)
✓ Exportiert: RI_1009_1.txt (9 Messungen)
✓ Exportiert: RI_1009_2.txt (12 Messungen)

✓ Export abgeschlossen!
✓ 5 Dateien erstellt
✓ Verzeichnis: test_data/Umwandlung_ltop-jag3d/jag3d_export


### 1.60. Zusammenführen DS/ZD-Messungen

In [13]:
import os

## Zusammenführung aller ZD und DS Dateien

# Archiv-Ordner für einzelne Dateien erstellen
archive_dir = os.path.join(output_directory, 'individual_DS-ZD_files')
os.makedirs(archive_dir, exist_ok=True)
# ZD-Dateien finden
zd_files = [f for f in exported_files if 'ZD' in os.path.basename(f)]
if zd_files:
    zd_output = os.path.join(output_directory, 'ZD_Messungen.txt')
    # Gruppennamen aus Dateinamen extrahieren
    zd_groups = [os.path.basename(f).replace('.txt', '') for f in zd_files]
    with open(zd_output, 'w', encoding='utf-8') as outfile:
        # Erweiterter Header mit Gruppeninfo
        outfile.write("# JAG3D Import File - Alle ZD Messungen\n")
        outfile.write("# Zusammengeführt aus mehreren ZD-Dateien\n")
        outfile.write(f"# Enthält Daten aus {len(zd_files)} Gruppen:\n")
        for group in zd_groups:
            outfile.write(f"#   - {group}\n")
        outfile.write("# Format: <start point> <end point> <instrument height> <reflector height> <observation value> <uncertainty>\n")
        outfile.write("\n")
        # Alle ZD-Dateien durchlaufen und Inhalt kopieren (ohne Header)
        for zd_file in zd_files:
            with open(zd_file, 'r', encoding='utf-8') as infile:
                lines = infile.readlines()
                for line in lines:
                    if not line.startswith('#') and line.strip():
                        outfile.write(line)
    # ZD-Dateien in Archiv-Ordner verschieben (statt löschen)
    for zd_file in zd_files:
        filename = os.path.basename(zd_file)
        new_path = os.path.join(archive_dir, filename)
        os.rename(zd_file, new_path)
# DS-Dateien finden
ds_files = [f for f in exported_files if 'DS' in os.path.basename(f)]
if ds_files:
    ds_output = os.path.join(output_directory, 'DS_Messungen.txt')
    # Gruppennamen aus Dateinamen extrahieren
    ds_groups = [os.path.basename(f).replace('.txt', '') for f in ds_files]
    with open(ds_output, 'w', encoding='utf-8') as outfile:
        # Erweiterter Header mit Gruppeninfo
        outfile.write("# JAG3D Import File - Alle DS Messungen\n")
        outfile.write("# Zusammengeführt aus mehreren DS-Dateien\n")
        outfile.write(f"# Enthält Daten aus {len(ds_files)} Gruppen:\n")
        for group in ds_groups:
            outfile.write(f"#   - {group}\n")
        outfile.write("# Format: <start point> <end point> <instrument height> <reflector height> <observation value> <uncertainty>\n")
        outfile.write("\n")
        # Alle DS-Dateien durchlaufen und Inhalt kopieren (ohne Header)
        for ds_file in ds_files:
            with open(ds_file, 'r', encoding='utf-8') as infile:
                lines = infile.readlines()
                for line in lines:
                    if not line.startswith('#') and line.strip():
                        outfile.write(line)
    # DS-Dateien in Archiv-Ordner verschieben (statt löschen)
    for ds_file in ds_files:
        filename = os.path.basename(ds_file)
        new_path = os.path.join(archive_dir, filename)
        os.rename(ds_file, new_path)

## 2.0. Automatische Umwandlung

### 2.10. Helper Function

In [1]:
def ltop2jag3d(filepath:str, output_directory:str):

    import os

    ## Datei Einlesen
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except UnicodeDecodeError:
        with open(filepath, 'r', encoding='latin-1') as f:
            lines = f.readlines()


    ## Filterung der unnötigen Zeilen (Kommentar, Titelzeile)
    filtered_lines = []
    for line in lines:
        line = line.strip()
        if line and not line.startswith('**') and not line.startswith('$$'):
            filtered_lines.append(line)


    ## Aufteilen in Stationsblöcke
    stations = []
    current_station = None

    for line in filtered_lines:
        if line.startswith('ST'):
            # Neue Station gefunden - vorherige Station speichern
            if current_station is not None:
                stations.append(current_station)

            # Station parsen: ST<Punktname>    <Instrumentenhöhe>
            parts = line.split()
            station_name = parts[0][2:]  # ST entfernen
            instrument_height = float(parts[1]) if len(parts) > 1 else None

            current_station = {
                'station_name': station_name,
                'instrument_height': instrument_height,
                'measurements': []
            }

        elif current_station is not None and line:
            # Messung zur aktuellen Station hinzufügen
            current_station['measurements'].append(line)

    # Letzte Station hinzufügen
    if current_station is not None:
        stations.append(current_station)


    # Messungen parsen und Gruppieren
    station_measurement_groups = {}

    for station_index, station in enumerate(stations):
        station_name = station['station_name']
        station_height = station['instrument_height']

        for measurement_line in station['measurements']:
            # Messzeile parsen
            measurement_type = measurement_line[:2]
            target_point = measurement_line[2:12].strip()
            measurement_value = float(measurement_line[24:36].strip())
            mf = measurement_line[36:42].strip()
            instrument_height_pt = measurement_line[46:52].strip()
            target_height_pt = measurement_line[52:58].strip()

            # Falls keine explizite Instrumentenhöhe gefunden und DS/ZD, Station-Höhe verwenden
            if len(instrument_height_pt) == 0 and station_height is not None and measurement_type not in ["RI"]:
                instrument_height_pt = station_height

            ## mF Umwandeln von mm in m
            if measurement_type in ["DS"] and len(mf) > 0:
                mf = float(mf) / 1_000

            ## mF Umwandeln von cc in gon
            if measurement_type in ["ZD", "RI"] and len(mf) > 0:
                mf = float(mf) / 10_000

            # NEUE FILTERUNG: Eliminierte Messungen überspringen
            if isinstance(mf, (int, float)) and (mf == 9.999 or mf == 0.9999):
                continue  # Diese Messung überspringen - wird nicht zu measurement_groups hinzugefügt
            
            # Bestimme den Gruppierungsschlüssel basierend auf Messtyp
            if measurement_type == "RI":
                # RI: Getrennt nach Aufstellung (Index)
                group_key = f"{station_name}_{station_index}"
            elif measurement_type == "DS":
                # DS: Eine eigene Gruppe für alle DS-Messungen
                group_key = f"DS_{station_name}"
            elif measurement_type == "ZD":
                # ZD: Eine eigene Gruppe für alle ZD-Messungen
                group_key = f"ZD_{station_name}"
            else:
                # Fallback für andere Messtypen
                group_key = f"{measurement_type}_{station_name}"

            # Gruppe initialisieren falls nötig
            if group_key not in station_measurement_groups:
                station_measurement_groups[group_key] = {}

            # Messtyp in der Gruppe initialisieren falls nötig
            if measurement_type not in station_measurement_groups[group_key]:
                station_measurement_groups[group_key][measurement_type] = []

            measurement_data = {
                'standpunkt': station_name,           # Standpunkt
                'anschlusspunkt': target_point,       # Anschlusspunkt
                'instrumentenhoehe': instrument_height_pt,  # Instrumentenhöhe
                'zielhoehe': target_height_pt,          # Zielhöhe
                'messwert': measurement_value,       # Messwert
                'standardabweichung': mf, # Standardabweichung
            }

            station_measurement_groups[group_key][measurement_type].append(measurement_data)

    
    ## Exportieren der gruppierten Messungen ins JAG3D-Format
    os.makedirs(output_directory, exist_ok=True)
    
    exported_files = []
    filenames = []
    length_files = []
    
    for group_key, measurement_types in station_measurement_groups.items():
        for measurement_type, measurements in measurement_types.items():
            if not measurements:
                continue
            
            # Korrigierte Dateiname-Erstellung - keine Dopplung bei DS/ZD
            if 'DS_' in group_key or 'ZD_' in group_key:
                filename = f"{group_key}.txt"  # Nur group_key verwenden
            else:
                filename = f"{measurement_type}_{group_key}.txt"  # Bei RI weiterhin mit Messtyp

            filepath = os.path.join(output_directory, filename)
            
            with open(filepath, 'w', encoding='utf-8') as f:
                # Header schreiben
                f.write(f"# JAG3D Import File - {measurement_type} Messungen\n")
                f.write(f"# Gruppe: {group_key}\n")
                f.write(f"# Format: <start point> <end point> <instrument height> <reflector height> <observation value> <uncertainty>\n")
                f.write(f"# Generiert aus LTOP mes-Datei\n")
                f.write("\n")
                
                for measurement in measurements:
                    # Werte extrahieren - keine Defaults setzen
                    start_point = measurement['standpunkt']
                    end_point = measurement['anschlusspunkt'] 
                    
                    # Instrumentenhöhe: Nur verwenden wenn vorhanden und nicht leer
                    instrument_height = measurement['instrumentenhoehe']
                    if instrument_height == '' or instrument_height is None:
                        instrument_height = ''
                    else:
                        instrument_height = str(instrument_height)
                    
                    # Zielhöhe: Nur verwenden wenn vorhanden und nicht leer
                    reflector_height = measurement['zielhoehe']
                    if reflector_height == '' or reflector_height is None:
                        reflector_height = ''
                    else:
                        reflector_height = str(reflector_height)
                    
                    # Messwert
                    observation_value = str(measurement['messwert'])
                    
                    # Unsicherheit: Nur verwenden wenn vorhanden und nicht leer
                    uncertainty = measurement['standardabweichung']
                    if uncertainty == '' or uncertainty is None:
                        uncertainty = ''
                    else:
                        uncertainty = str(uncertainty)
                    
                    # Zeile zusammenstellen (Tabulator-getrennt, leere Felder bleiben leer)
                    line = start_point + '\t' + end_point + '\t' + instrument_height + '\t' + reflector_height + '\t' + observation_value + '\t' + uncertainty + '\n'
                    f.write(line)
            
            exported_files.append(filepath)
            filenames.append(filename)
            length_files.append(len(measurements))


    ## Zusammenführung aller ZD und DS Dateien

    # Archiv-Ordner für einzelne Dateien erstellen
    archive_dir = os.path.join(output_directory, 'individual_DS-ZD_files')
    os.makedirs(archive_dir, exist_ok=True)

    # ZD-Dateien finden
    zd_files = [f for f in exported_files if 'ZD' in os.path.basename(f)]
    if zd_files:
        zd_output = os.path.join(output_directory, 'ZD_Messungen.txt')

        # Gruppennamen aus Dateinamen extrahieren
        zd_groups = [os.path.basename(f).replace('.txt', '') for f in zd_files]

        with open(zd_output, 'w', encoding='utf-8') as outfile:
            # Erweiterter Header mit Gruppeninfo
            outfile.write("# JAG3D Import File - Alle ZD Messungen\n")
            outfile.write("# Zusammengeführt aus mehreren ZD-Dateien\n")
            outfile.write(f"# Enthält Daten aus {len(zd_files)} Gruppen:\n")
            for group in zd_groups:
                outfile.write(f"#   - {group}\n")
            outfile.write("# Format: <start point> <end point> <instrument height> <reflector height> <observation value> <uncertainty>\n")
            outfile.write("\n")

            # Alle ZD-Dateien durchlaufen und Inhalt kopieren (ohne Header)
            for zd_file in zd_files:
                with open(zd_file, 'r', encoding='utf-8') as infile:
                    lines = infile.readlines()
                    for line in lines:
                        if not line.startswith('#') and line.strip():
                            outfile.write(line)

        # ZD-Dateien in Archiv-Ordner verschieben (statt löschen)
        for zd_file in zd_files:
            filename = os.path.basename(zd_file)
            new_path = os.path.join(archive_dir, filename)
            os.rename(zd_file, new_path)

    # DS-Dateien finden
    ds_files = [f for f in exported_files if 'DS' in os.path.basename(f)]
    if ds_files:
        ds_output = os.path.join(output_directory, 'DS_Messungen.txt')

        # Gruppennamen aus Dateinamen extrahieren
        ds_groups = [os.path.basename(f).replace('.txt', '') for f in ds_files]

        with open(ds_output, 'w', encoding='utf-8') as outfile:
            # Erweiterter Header mit Gruppeninfo
            outfile.write("# JAG3D Import File - Alle DS Messungen\n")
            outfile.write("# Zusammengeführt aus mehreren DS-Dateien\n")
            outfile.write(f"# Enthält Daten aus {len(ds_files)} Gruppen:\n")
            for group in ds_groups:
                outfile.write(f"#   - {group}\n")
            outfile.write("# Format: <start point> <end point> <instrument height> <reflector height> <observation value> <uncertainty>\n")
            outfile.write("\n")

            # Alle DS-Dateien durchlaufen und Inhalt kopieren (ohne Header)
            for ds_file in ds_files:
                with open(ds_file, 'r', encoding='utf-8') as infile:
                    lines = infile.readlines()
                    for line in lines:
                        if not line.startswith('#') and line.strip():
                            outfile.write(line)

        # DS-Dateien in Archiv-Ordner verschieben (statt löschen)
        for ds_file in ds_files:
            filename = os.path.basename(ds_file)
            new_path = os.path.join(archive_dir, filename)
            os.rename(ds_file, new_path)

    # return exported_files, filenames, length_files

### 2.20. Aufrufen der Funktion

In [2]:
mes_file = "test_data/Umwandlung_ltop-jag3d/20250422_Gr3_Messungen_SatzGetrennt_sort_korr.mes"
output_ordner = "test_data/Umwandlung_ltop-jag3d/jag3d_export_live"

# exported_files, filenames, length_files = ltop2jag3d(mes_file, output_ordner)

ltop2jag3d(mes_file, output_ordner)

# print(f"✓ Export abgeschlossen! / {len(exported_files)} Dateien erstellt\n")
# for i in range(len(exported_files)):
#     print(f"Datei: {filenames[i]}, ({length_files[i]} Messungen)")

In [5]:
mes_file = "test_data/Umwandlung_ltop-jag3d/20251015_Messdaten_Netzausgleich_korr.mes"
output_ordner = "test_data/Umwandlung_ltop-jag3d/01_Messungen_Madrisa"

ltop2jag3d(mes_file, output_ordner)